In [2]:
import sys
sys.path.append(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq")

from TCC.utils.constantes import *
import matplotlib.pyplot as plt

## Transaction Volume ETH
- Transaction volume: mensura o volume total movimentado em ETH na semana.

- A Ethereum representa o "Risco Tecnológico" (Smart Contracts, DeFi, NFTs). Quando o volume na ETH explode, significa que o mercado está buscando utilidade e risco, saindo da segurança do Bitcoin. É o principal indicador de "Altseason" real (baseada em uso, não só especulação).

- Fonte: https://academy.santiment.net/metrics/transaction-volume/#available-assets

### HIPÓTESES
## Hipótese 1: Era Varejo (Pré-2020)
A Rivalidade Especulativa (Narrativa do "Flippening"). Neste período, o volume na rede Ethereum era impulsionado primariamente por ICOs (Initial Coin Offerings) e pela tese de que o ETH ultrapassaria o Bitcoin. Hipotetiza-se uma correlação instável ou negativa com o preço do Bitcoin. Picos de atividade no Ethereum frequentemente coincidiam com fluxos de saída do Bitcoin (Capital Rotation), onde o preço do líder estagnava enquanto a liquidez especulativa migrava para o "novo paradigma", gerando divergência de preços.

## Hipótese 2: Era Institucional (Pós-2020)
Correlação de Risco Sistémico (Risk-On Beta). Com a consolidação do setor (DeFi Summer e NFTs), o Ethereum estabeleceu-se como uma plataforma de utilidade tecnológica, e não apenas um rival monetário. Hipotetiza-se uma forte correlação positiva com o preço do Bitcoin. Alta atividade na rede ETH sinaliza um ambiente de apetite ao risco e liquidez saudável no ecossistema cripto. Instituições tratam ambos os ativos como proxies de crescimento tecnológico; portanto, se a rede Ethereum está ativa e valorizando, o Bitcoin tende a seguir a mesma tendência de alta (Sympathetic Move).

### TRATAMENTO
- Volume financeiro total transacionado na rede Ethereum. Recomenda-se usar o Logaritmo do volume (para normalizar a escala exponencial de crescimento) e depois a Diferença Diária para capturar choques de atividade.
- Log-Retorno: Para capturar o choque de atividade. "Hoje a rede Ethereum movimentou 50% mais que ontem

In [14]:
df_eth_volume = (pd.read_csv(rf"raw/2015_transactionVolumeEth.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['Date'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                        .rename(columns={'Transaction Volume (ETH)': 'Transaction_Volume_ETH',
                                         'Transaction Volume USD (ETH)': 'Transaction_Volume_USD_ETH'}) 

                        [['Data_UTC', 'Transaction_Volume_ETH']] 
 
)

df_eth_volume_diff =(
    df_periodo
        .merge(df_eth_volume, how='left', on='Data_UTC')
        .assign(Transaction_Volume_ETH = lambda df: df['Transaction_Volume_ETH'].replace(0, np.nan))
        .assign(Transaction_Volume_ETH = lambda df: df['Transaction_Volume_ETH'].fillna(method='ffill'))
        .assign(ETH_Vol_Log_Ret = lambda df: np.log(df['Transaction_Volume_ETH']) - np.log(df['Transaction_Volume_ETH'].shift(1)))


        .query("Data_UTC > '2017-01-02'")
        [['Data_UTC','Transaction_Volume_ETH','ETH_Vol_Log_Ret']]

)
df_eth_volume_diff

# print_dataframe_info(df_eth_volume_diff, "Ethereum Transaction Volume")

/var/folders/h1/_3z4z04x4j17zlfj224w53hr0000gn/T/ipykernel_3848/4151286280.py:17: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .assign(Transaction_Volume_ETH = lambda df: df['Transaction_Volume_ETH'].fillna(method='ffill'))


,Data_UTC,Transaction_Volume_ETH,ETH_Vol_Log_Ret
3,2017-01-03,5.474942e+06,0.608957
4,2017-01-04,8.673430e+06,0.460083
5,2017-01-05,8.854667e+06,0.020680
6,2017-01-06,7.981456e+06,-0.103824
7,2017-01-07,4.804753e+06,-0.507515
...,...,...,...
3131,2025-07-28,4.279742e+06,0.434920
3132,2025-07-29,3.429451e+06,-0.221492
3133,2025-07-30,3.542731e+06,0.032498
3134,2025-07-31,3.746258e+06,0.055860


## ETH Exchange Supply (% of Total Supply) 
- Variação diária da porcentagem de Ethereum mantida em corretoras.
- O Ethereum nas exchanges tem apenas uma função principal: Liquidez para Venda. O Ethereum fora das exchanges tem utilidade real (DeFi, NFTs, Staking). Portanto, essa métrica é o inverso da "Saúde do Ecossistema Cripto". Já na era institucional, espera-se que a concentracao de ETH nas exchanges nao impacta tao negativamente o seu preco, devido a natureza das compras OTC

- Fonte: https://academy.santiment.net/metrics/supply-on-or-outside-exchanges/#definition

### HIPÓTESES
## Hipótese 1: Era Varejo (Pré-2020)
O Sinal de Pânico Imediato. A infraestrutura de mercado era dominada por vendas no mercado à vista (Spot Selling) por investidores individuais. Hipotetiza-se uma causalidade negativa direta e rápida.

Inflow (Entrada): O envio de moedas para exchanges era quase invariavelmente um prelúdio de venda (Dump). O aumento do saldo em exchange causava queda imediata no preço do Bitcoin devido à falta de profundidade do livro de ofertas para absorver a liquidez.

## Hipótese 2: Era Institucional (Pós-2020)
Absorção OTC e Amortecimento de Impacto. A dinâmica de custódia mudou com a entrada de mesas de balcão (OTC Desks) e Market Makers. Hipotetiza-se que o poder preditivo de curto prazo dos "Inflows" sobre a queda de preço diminuiu drasticamente.

O aumento de saldo em exchange pode refletir preparação de liquidez para criação de mercado ou custódia institucional transitória, e não necessariamente pressão de venda direcional. Grandes ordens de venda são executadas fora do livro público (Dark Pools), tornando a métrica de Exchange Supply menos ruidosa para crashes, mas ainda relevante para tendências de longo prazo (onde a saída de moedas/Outflow sustenta a alta do preço por choque de oferta).

### TRATAMENTO
- O Diff vai mostrar exatamente quantos % do Supply total entraram ou saíram das exchanges nas últimas 24h. Isso é uma proxy perfeita para Net Flow.
- A diferença simples captura o fluxo líquido (Net Flow) real de ativos. Valores positivos indicam injeção de liquidez para venda; negativos indicam retirada para custódia ou DeFi.

In [27]:
df_supply_top_wallets = (pd.read_csv(rf"raw/2015_supply_held_by_exchanges_top_wallets_2.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['Date'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                        .rename(columns={'Supply held by top exchange addresses (ETH)': 'Supply_Held_Top_Exchange_Addresses_ETH',
                                         f'Supply on Exchanges (as % of total supply) (ETH)': 'Supply_on_Exchanges_Percent_Total_Supply_ETH',
                                         f'Supply held by top addresses (as % of total supply) (ETH)': 'Supply_Held_Top_Addresses_Percent_Total_Supply_ETH'}) 

                        [['Data_UTC', 'Supply_on_Exchanges_Percent_Total_Supply_ETH']] 
 
)
# df_eth_volume
df_in_out_flow =(
    df_periodo
        .merge(df_supply_top_wallets, how='left', on='Data_UTC')
        .assign(ETH_Supply_Flow_Diff = lambda df: df['Supply_on_Exchanges_Percent_Total_Supply_ETH'].diff())


        .query("Data_UTC > '2016-12-31'")
        # [['Data_UTC','Transaction_Volume_ETH','ETH_Vol_Log_Ret']]

)
df_in_out_flow

# print_dataframe_info(df_in_out_flow, "Supply on Exchanges and Top Wallets Ethereum")

,Data_UTC,Supply_on_Exchanges_Percent_Total_Supply_ETH,ETH_Supply_Flow_Diff
1,2017-01-01,24.118518,-0.108907
2,2017-01-02,24.122218,0.003700
3,2017-01-03,24.063647,-0.058571
4,2017-01-04,24.181199,0.117552
5,2017-01-05,24.196631,0.015431
...,...,...,...
3131,2025-07-28,8.181490,0.008197
3132,2025-07-29,8.162358,-0.019132
3133,2025-07-30,8.181616,0.019258
3134,2025-07-31,8.008848,-0.172768


## VOLUME SOCIAL DA PALAVRA "ETH" E RELACIONADOS NAS REDES SOCIAS
- Contagem total de documentos únicos (mensagens Telegram, posts Reddit, tweets) que mencionam "ETH" pelo menos uma vez. Como a métrica é baseada em documentos, ela mede a viralidade da discussão e a participação da massa (Varejo).

- No mercado de criptoativos, "Atenção precede o Preço". O Social Volume funciona como um proxy de FOMO (Fear Of Missing Out). Quando o número de documentos explode, significa que pessoas que normalmente não postam estão falando sobre isso.

- Fonte1: https://academy.santiment.net/metrics/social-volume/
- Fonte2: https://academy.santiment.net/metrics/social-dominance/
- Fonte3: https://academy.santiment.net/metrics/social-volume-ai/

### HIPÓTESES
## Hipótese 1: A Era do Contágio Social (2017–2020)
Hipótese da Viralidade Linear. Durante a 'Era Varejo', postula-se que o Volume Social atuava como uma proxy direta de demanda para a entrada de capital não sofisticado. Dada a predominância de investidores individuais movidos a emoção, picos de menções em redes sociais geravam um efeito de manada (Herd Behavior) imediato.

Expectativa Estatística: Espera-se uma Causalidade de Granger positiva e linear: Aumento do Social Volume -> Aumento do Preço do Bitcoin (FOMO impulsionando o ativo). O "Buzz" precedia a valorização.

## Hipótese 2: A Era da Eficiência Institucional (Pós-2020)
Hipótese do Ruído e Sinal Contrário. Na 'Era Institucional', propõe-se que a relação entre "Hype" e Preço tornou-se não-linear ou inversa nos extremos. A entrada de capital inteligente (Smart Money) introduz modelos que filtram o ruído das redes sociais.

Expectativa Estatística: Espera-se uma perda de significância preditiva linear (redução do SHAP Value) ou uma inversão de sinal.

O "Hype" excessivo no Twitter agora é interpretado por algoritmos institucionais como Sinal de Topo (exuberância irracional), gatilhando vendas. Portanto, um volume social extremo pode preceder quedas de preço (correção), ao contrário da era anterior onde garantia alta.

### TRATAMENTO
- Social Dominance: Primeira Diferença (Diff)
- Social Volume: Log-Retorno

In [41]:
df_social_volume = (pd.read_csv(rf"raw/eth_social_metrics.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['Date'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                        .rename(columns={'Social Dominance (ETH)': 'Social_Dominance_ETH',
                                         'Social Volume (ETH)': 'Social_Volume_ETH',
                                         'Social Volume AI (ETH)': 'Social_Volume_AI_ETH',
                                         'Trending Social Rank (ETH)': 'Trending_Social_Rank_ETH'}) 

                        [['Data_UTC', 'Social_Dominance_ETH', 'Social_Volume_ETH', 'Social_Volume_AI_ETH']] 
 
)
df_social_volume

df_social_metrics_eth =(
    df_periodo
        .merge(df_social_volume, how='left', on='Data_UTC')
        .assign(Social_Dom_Diff = lambda df: df['Social_Dominance_ETH'].diff())
        .assign(Social_Vol_Log_Ret = lambda df: np.log(df['Social_Volume_ETH']) - np.log(df['Social_Volume_ETH'].shift(1)))

        .query("Data_UTC > '2016-12-31'")
        [['Data_UTC','Social_Dominance_ETH','Social_Dom_Diff','Social_Volume_ETH','Social_Vol_Log_Ret']]

)
df_social_metrics_eth

# print_dataframe_info(df_social_metrics_eth, "Social Metrics Ethereum")

,Data_UTC,Social_Dominance_ETH,Social_Dom_Diff,Social_Volume_ETH,Social_Vol_Log_Ret
1,2017-01-01,7.787378,0.563079,283.0,0.088619
2,2017-01-02,7.352515,-0.434863,380.0,0.294724
3,2017-01-03,8.032541,0.680026,453.0,0.175721
4,2017-01-04,9.152380,1.119839,575.0,0.238478
5,2017-01-05,7.715741,-1.436639,502.0,-0.135770
...,...,...,...,...,...
3131,2025-07-28,12.745914,3.226667,2293.0,0.507053
3132,2025-07-29,12.478293,-0.267621,1939.0,-0.167689
3133,2025-07-30,12.497530,0.019237,1866.0,-0.038375
3134,2025-07-31,11.658707,-0.838823,1488.0,-0.226364
